# Neural Collaborative Filtering: accept/decline prediction

Trains a small embedding-based model to predict whether a mentorship/connection request between two stakeholders would be **ACCEPTED**, using the synthetic seed dataset (`prisma/data/stakeholders.csv`, `prisma/data/interactions.csv`) from the [Intelligent Stakeholder Matching](https://github.com/K6EDWIN/Intelligent-stakeholder-matching) project.

**This repo is private.** The data-loading cell below needs either a `GITHUB_TOKEN` secret (Colab Secrets panel) or you'll be prompted to upload the two CSVs manually -- see that cell's comments for details.

This is a standalone research/evaluation artifact -- it does **not** feed into the live app's recommendation ranking (see `docs/ML_MODEL.md` in the repo for why, and what it found). Run all cells top to bottom; checkpoints and exported results go to Google Drive.

## 1. Setup: mount Drive, get the data (private repo -- see next cell)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CHECKPOINT_DIR = "/content/drive/MyDrive/ism-checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoints will be saved to: {CHECKPOINT_DIR}")

In [ ]:
# This repo is PRIVATE, so an anonymous git clone over HTTPS will
# silently fail here (no error surfaced in some Colab versions -- it just
# doesn't produce the directory) and the next cell's pd.read_csv() will
# raise FileNotFoundError. Two ways to get the data in:
#
# Option A (recommended) -- a GitHub token stored in Colab's Secrets panel
# (key icon in the left sidebar), named GITHUB_TOKEN, with at least
# repo read access, shared with this notebook. Keeps the notebook's data
# in sync with whatever's on main automatically, every run.
#
# Option B (fallback, no setup required) -- if no token is available,
# this cell prompts you to upload stakeholders.csv and interactions.csv
# directly (from prisma/data/ in your local checkout, or downloaded from
# GitHub). Works immediately but won't auto-update on a later run.

import os
import shutil

REPO_URL = "github.com/K6EDWIN/Intelligent-stakeholder-matching.git"
CLONE_DIR = "/content/Intelligent-stakeholder-matching"
ARTIFACT_DIR = "/content/artifacts"
os.makedirs(ARTIFACT_DIR, exist_ok=True)

DATA_DIR = None

try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None

if token:
    shutil.rmtree(CLONE_DIR, ignore_errors=True)
    clone_url = "https://" + token + "@" + REPO_URL
    result = os.system("git clone --depth 1 " + clone_url + " " + CLONE_DIR + " > /tmp/clone.log 2>&1")
    candidate = CLONE_DIR + "/prisma/data"
    if result == 0 and os.path.exists(candidate + "/stakeholders.csv"):
        DATA_DIR = candidate
        print("Cloned via GITHUB_TOKEN secret. DATA_DIR =", DATA_DIR)
    else:
        print("Clone with GITHUB_TOKEN failed or files missing -- see /tmp/clone.log. Falling back to manual upload.")
else:
    print("No GITHUB_TOKEN secret found (Colab Secrets panel, left sidebar). Falling back to manual upload.")

if DATA_DIR is None:
    from google.colab import files
    DATA_DIR = "/content/data"
    os.makedirs(DATA_DIR, exist_ok=True)
    for needed in ("stakeholders.csv", "interactions.csv"):
        target = DATA_DIR + "/" + needed
        if not os.path.exists(target):
            print("")
            print("Upload " + needed + " (from prisma/data/" + needed + " in your local checkout):")
            uploaded = files.upload()
            for name, content in uploaded.items():
                with open(DATA_DIR + "/" + name, "wb") as out_f:
                    out_f.write(content)
    print("")
    print("Using uploaded files. DATA_DIR =", DATA_DIR)

assert os.path.exists(DATA_DIR + "/stakeholders.csv"), "stakeholders.csv still not found in " + DATA_DIR
assert os.path.exists(DATA_DIR + "/interactions.csv"), "interactions.csv still not found in " + DATA_DIR

## 2. Load and preprocess data

Training set = every interaction row where `status != PENDING` (label = 1 if `ACCEPTED`, else 0). `request_type` becomes a one-hot feature alongside the two user embeddings.

In [ ]:
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.decomposition import PCA

torch.manual_seed(42)
np.random.seed(42)

stakeholders = pd.read_csv(f"{DATA_DIR}/stakeholders.csv")
interactions = pd.read_csv(f"{DATA_DIR}/interactions.csv")

user_ids = stakeholders["user_id"].tolist()
user_id_to_idx = {uid: i for i, uid in enumerate(user_ids)}
user_role = dict(zip(stakeholders["user_id"], stakeholders["role"]))
num_users = len(user_ids)
print(f"num_users: {num_users}")

In [ ]:
labeled = interactions[interactions["status"] != "PENDING"].copy()
labeled["label"] = (labeled["status"] == "ACCEPTED").astype(int)
labeled["sender_idx"] = labeled["requester_id"].map(user_id_to_idx)
labeled["recipient_idx"] = labeled["recipient_id"].map(user_id_to_idx)
labeled["is_mentorship"] = (labeled["request_type"] == "MENTORSHIP").astype(int)

assert labeled["sender_idx"].isna().sum() == 0
assert labeled["recipient_idx"].isna().sum() == 0

# Data-sparsity check -- this matters a lot for whether embedding-based CF
# can generalize at all (see the note at the bottom of this notebook).
pair_counts = labeled.groupby(["sender_idx", "recipient_idx"]).size()
print(f"labeled examples: {len(labeled)}, base accept rate: {labeled['label'].mean():.3f}")
print(f"unique (sender,recipient) pairs: {len(pair_counts)} / {len(labeled)} rows")
print(f"pairs seen more than once: {(pair_counts > 1).sum()}")

user_appearances = pd.concat([labeled["sender_idx"], labeled["recipient_idx"]]).value_counts()
print(f"appearances per user -- median: {user_appearances.median()}, min: {user_appearances.min()}, max: {user_appearances.max()}")

In [ ]:
train_val_df, test_df = train_test_split(labeled, test_size=0.2, random_state=42, stratify=labeled["label"])
train_df, val_df = train_test_split(train_val_df, test_size=0.2, random_state=42, stratify=train_val_df["label"])
print(f"train: {len(train_df)}, val: {len(val_df)}, test: {len(test_df)}")

def to_tensors(df):
    sender = torch.tensor(df["sender_idx"].values, dtype=torch.long)
    recipient = torch.tensor(df["recipient_idx"].values, dtype=torch.long)
    is_mentorship = torch.tensor(df["is_mentorship"].values, dtype=torch.float32)
    request_type_onehot = torch.stack([is_mentorship, 1 - is_mentorship], dim=1)
    label = torch.tensor(df["label"].values, dtype=torch.float32)
    return sender, recipient, request_type_onehot, label

train_t = to_tensors(train_df)
val_t = to_tensors(val_df)
test_t = to_tensors(test_df)

## 3. Model

Small NCF architecture, intentionally simple given ~150 users / ~370 labeled examples -- a bigger model would just overfit faster. One shared embedding table (looked up by both sender and receiver index) feeding a small MLP head.

In [ ]:
class NCF(nn.Module):
    def __init__(self, num_users, embedding_dim=8, hidden_dim=16):
        super().__init__()
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embedding_dim * 2 + 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, sender_idx, recipient_idx, request_type_onehot):
        s = self.user_embedding(sender_idx)
        r = self.user_embedding(recipient_idx)
        x = torch.cat([s, r, request_type_onehot], dim=1)
        return torch.sigmoid(self.mlp(x)).squeeze(1)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")

model = NCF(num_users, embedding_dim=8, hidden_dim=16).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-2)
loss_fn = nn.BCELoss()

## 4. Training

Full-batch (the dataset is tiny enough that mini-batching adds nothing), with early stopping on validation loss to fight overfitting, and a checkpoint saved to Google Drive whenever validation loss improves.

In [ ]:
train_sender, train_recipient, train_type, train_label = (t.to(device) for t in train_t)
val_sender, val_recipient, val_type, val_label = (t.to(device) for t in val_t)

best_val_loss = float("inf")
best_epoch = 0
EPOCHS = 300
patience = 30
epochs_since_improvement = 0
train_losses, val_losses = [], []

for epoch in range(EPOCHS):
    model.train()
    optimizer.zero_grad()
    pred = model(train_sender, train_recipient, train_type)
    loss = loss_fn(pred, train_label)
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        val_pred = model(val_sender, val_recipient, val_type)
        val_loss = loss_fn(val_pred, val_label).item()

    train_losses.append(loss.item())
    val_losses.append(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        epochs_since_improvement = 0
        torch.save(model.state_dict(), f"{CHECKPOINT_DIR}/best_model.pt")
    else:
        epochs_since_improvement += 1

    if (epoch + 1) % 25 == 0:
        print(f"epoch {epoch+1}: train_loss={loss.item():.4f} val_loss={val_loss:.4f} (best={best_val_loss:.4f}@{best_epoch})")

    if epochs_since_improvement >= patience:
        print(f"early stopping at epoch {epoch+1}, best epoch was {best_epoch}")
        break

torch.save(model.state_dict(), f"{CHECKPOINT_DIR}/final_model.pt")
model.load_state_dict(torch.load(f"{CHECKPOINT_DIR}/best_model.pt"))
model.eval()
print(f"\nLoaded best checkpoint (epoch {best_epoch}) for evaluation.")

## 5. Evaluation

In [ ]:
test_sender, test_recipient, test_type, test_label = (t.to(device) for t in test_t)
with torch.no_grad():
    test_pred_proba = model(test_sender, test_recipient, test_type).cpu().numpy()
test_pred = (test_pred_proba >= 0.5).astype(int)
test_true = test_label.cpu().numpy().astype(int)

acc = accuracy_score(test_true, test_pred)
prec = precision_score(test_true, test_pred, zero_division=0)
rec = recall_score(test_true, test_pred, zero_division=0)
f1 = f1_score(test_true, test_pred, zero_division=0)
auc = roc_auc_score(test_true, test_pred_proba)
base_rate = max(labeled["label"].mean(), 1 - labeled["label"].mean())

print("--- Test metrics (best-val-epoch model) ---")
print(f"accuracy:  {acc:.4f}")
print(f"precision: {prec:.4f}")
print(f"recall:    {rec:.4f}")
print(f"f1:        {f1:.4f}")
print(f"roc_auc:   {auc:.4f}")
print(f"majority-class baseline: {base_rate:.4f}")

In [ ]:
# Precision@5 ranking eval -- for each test-set sender, rank all other users
# by predicted accept-probability and check whether their true ACCEPTED
# counterparts land near the top. Same style of metric the project's
# evaluation work (issue #11, Precision@K/Recall@K/NDCG) needs.
precision_at_5_scores = []
with torch.no_grad():
    for sender_idx in test_df["sender_idx"].unique():
        true_accepted = set(test_df[(test_df["sender_idx"] == sender_idx) & (test_df["label"] == 1)]["recipient_idx"])
        if not true_accepted:
            continue
        all_recipients = torch.arange(num_users, device=device)
        sender_batch = torch.full((num_users,), int(sender_idx), dtype=torch.long, device=device)
        type_batch = torch.tensor([[1.0, 0.0]] * num_users, device=device)
        scores = model(sender_batch, all_recipients, type_batch).cpu().numpy()
        top5 = set(np.argsort(-scores)[:5].tolist())
        hit = len(top5 & true_accepted) / min(5, len(true_accepted))
        precision_at_5_scores.append(hit)

precision_at_5 = float(np.mean(precision_at_5_scores)) if precision_at_5_scores else None
print(f"Precision@5 (ranking eval, {len(precision_at_5_scores)} users): {precision_at_5:.4f}" if precision_at_5 is not None else "no eligible test users")

## 6. Plots

In [ ]:
plt.figure(figsize=(7, 4.5))
plt.plot(train_losses, label="train loss")
plt.plot(val_losses, label="val loss")
plt.axvline(best_epoch, color="gray", linestyle="--", alpha=0.6, label=f"best epoch ({best_epoch})")
plt.xlabel("epoch")
plt.ylabel("BCE loss")
plt.title("NCF training/validation loss")
plt.legend()
plt.tight_layout()
plt.savefig(f"{ARTIFACT_DIR}/loss_curve.png", dpi=120)
plt.show()

In [ ]:
embeddings = model.user_embedding.weight.detach().cpu().numpy()
pca = PCA(n_components=2)
coords = pca.fit_transform(embeddings)
roles = [user_role[uid] for uid in user_ids]

plt.figure(figsize=(7, 6))
for role in sorted(set(roles)):
    idx = [i for i, r in enumerate(roles) if r == role]
    plt.scatter(coords[idx, 0], coords[idx, 1], label=role, alpha=0.7, s=30)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Learned user embeddings (PCA), colored by role")
plt.legend()
plt.tight_layout()
plt.savefig(f"{ARTIFACT_DIR}/embedding_space.png", dpi=120)
plt.show()

## 7. Export

Saves to both Google Drive (for permanence) and `/content/artifacts` (for downloading back into the repo's `ml/artifacts/`).

In [ ]:
metrics = {
    "num_users": num_users,
    "num_labeled_examples": len(labeled),
    "unique_sender_recipient_pairs": int(len(pair_counts)),
    "repeated_pairs": int((pair_counts > 1).sum()),
    "train_size": len(train_df),
    "val_size": len(val_df),
    "test_size": len(test_df),
    "base_accept_rate": float(labeled["label"].mean()),
    "majority_class_baseline_accuracy": float(base_rate),
    "best_epoch": int(best_epoch),
    "stopped_at_epoch": len(train_losses),
    "test_accuracy": float(acc),
    "test_precision": float(prec),
    "test_recall": float(rec),
    "test_f1": float(f1),
    "test_roc_auc": float(auc),
    "precision_at_5": precision_at_5,
    "embedding_dim": 8,
    "hidden_dim": 16,
    "weight_decay": 1e-2,
    "device_used": device,
}

for out_dir in (CHECKPOINT_DIR, ARTIFACT_DIR):
    with open(f"{out_dir}/metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)
    embeddings_dict = {uid: embeddings[idx].tolist() for uid, idx in user_id_to_idx.items()}
    with open(f"{out_dir}/embeddings.json", "w") as f:
        json.dump(embeddings_dict, f)

print(json.dumps(metrics, indent=2))
print(f"\nSaved metrics.json + embeddings.json to {CHECKPOINT_DIR} and {ARTIFACT_DIR}")
print("Download /content/artifacts/*.json and *.png and commit them into ml/artifacts/ in the repo if this run should replace the checked-in results.")

## Finding

The model does not beat the majority-class baseline (ROC-AUC ~0.5, essentially random). Every (sender, recipient) pair in the dataset appears **exactly once** -- there is no repeated co-occurrence signal for user embeddings to learn from. This was confirmed to be a genuine data-sparsity limit, not an implementation bug: an unregularized version of this same model fits training data to near-zero loss (it has the capacity to learn), it simply has nothing generalizable to learn from this interaction volume.

This empirically validates the live app's `src/lib/hybrid.js` design, which deliberately weights content-based similarity (alpha starts at 0.9) over collaborative signal when accepted-interaction counts are low -- exactly the regime this dataset is in. See `docs/ML_MODEL.md` in the repo for the full writeup.